# Automatic Detection Configuration

## Method

Pile driving impacts are detected in the time series using a threshold-based approach.
First, data are high-pass filtered to remove low-frequency noise.
The high-pass filter is a fourth-order Butterworth infinite impulse response filter with a cutoff frequency of 10 Hz.
A characteristic function based on the root-mean-square (RMS) of the filtered signal is computed to aid in detection of pile-driving impacts.
Given the filtered signal $y[n]$, the RMS over a moving window of length $M_1$ is defined as:
\begin{equation}
  y_\mathrm{RMS}[n] = \sqrt{ \frac{1}{M_1}\sum_{k=0}^{M_1-1} y[n-k]^2},
  \tag{1}
\end{equation}
where the window length $M_1$ is chosen to be 25 m/s and:
\begin{equation}
  M_1 = f_s T = \lfloor(\textrm{19100 samples/s})(\textrm{25 m/s})\rfloor = \textrm{488 samples}.
  \tag{2}
\end{equation}
To ensure the threshold used for detecting impulses is consistent across soft start and full power events, the characteristic function is obtained by normalizing Eq. [1] by its running mean computed over the moving window $M_2$:
\begin{equation}
  c[n] = \frac{y_\mathrm{RMS}[n]}{\frac{1}{M_2} \sum_{k=0}^{M_2} y_\mathrm{RMS}[n-k]},
  \tag{3}
\end{equation}
where the window length $M_2$ is chosen to be 5 s and:
\begin{equation}
  M_2 = f_s T = \lfloor(\textrm{19100 samples/s})(\textrm{5 s})\rfloor = \textrm{97656 samples}.
  \tag{4}
\end{equation}
A detection occurs when the characteristic function exceeds a user-defined threshold that is empirically set according to the acoustic environment at each hydrophone.
Triggering is suppressed for 0.75 s before or after a detection to prevent triggering on multipath arrivals.

## Configuration

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

### Import libraries & define helper functions

In [2]:
from pathlib import Path
import tomllib

import matplotlib.pyplot as plt
import matplotlib.style as mplstyle
import numpy as np
import pandas as pd
from tritonoa.data.reader import read_inventory
from tritonoa.data.time import TIME_PRECISION

from rodeo import plotting, utils

mplstyle.use("fast")

### Set paths to data and logs

In [3]:
EQUIPPATH = Path("config/experiment/equipment.csv")
EVENTSPATH = Path("config/events/events.csv")
INVPATH = Path("config/inventory.toml")
SAVEPATH = Path("results/events")

### Load inventory configuration

In [4]:
with open(INVPATH, "rb") as f:
    inv_config = tomllib.load(f)

### Select event

Set the `event_name` variable to the name of the event for which the workflow will be run. Possible selections are (all times UTC):

| Event | Start | Stop | Tetra | VLA1 | VLA2 | Pulses | Notes |
|-------|-------|------| ------| ---- | ---- | ------ | ----- |
| `Z01-A` | 2023-06-19T19:20 | 2023-06-19T21:10 | ❌ | ❌ | ❌ |  |  |
| `Z01-B` | 2023-06-19T22:25 | 2023-06-19T23:00 | ❌ | ❌ | ❌ |  |  |
| `Z01-C` | 2023-06-20T10:35 | 2023-06-20T10:50 | ❌ | ❌ | ❌ |  |  |
| `Z01-D` | 2023-06-20T13:15 | 2023-06-20T15:20 | ❌ | ❌ | ❌ |  |  |
| `A12-A` | 2023-07-02T17:04 | 2023-07-02T18:20 | 🟨 | 🟨 | 🟨 |  | Detector unreliable due to range |
| `A12-B` | 2023-07-02T19:50 | 2023-07-02T21:35 | ✅ | ✅ | ✅ | 2,608 |  |
| `A08`   | 2023-07-06T15:15 | 2023-07-06T18:20 | ✅ | ✅ | ✅ | 3,784 |  |
| `A10-A` | 2023-07-08T19:00 | 2023-07-08T20:40 | ✅ | ✅ | ✅ | 1,895 |  |
| `A10-B` | 2023-07-09T10:00 | 2023-07-09T11:30 | ✅ | ✅ | ✅ | 1,975 |  |
| `A03`   | 2023-07-12T19:00 | 2023-07-12T21:50 | ✅ | ✅ | ✅ | 4,200 |  |
| `A14`   | 2023-07-21T20:00 | 2023-07-21T23:50 | ❌ | ✅ | ✅ | 3,482 |  |
| `A11`   | 2023-07-23T20:40 | 2023-07-24T01:20 | ❌ | ✅ | ✅ | 4,879 |  |
| `A02`   | 2023-07-31T19:00 | 2023-07-31T22:20 | ❌ | ✅ | ✅ | 4,547 |  |
| `A15`   | 2023-08-01T20:15 | 2023-08-01T22:50 | ❌ | ✅ | ✅ | 3,828 |  |
| `A13`   | 2023-08-03T14:45 | 2023-08-03T17:40 | ❌ | ✅ | ✅ | 4,355 |  |
| `A01`   | 2023-08-04T15:20 | 2023-08-04T18:25 | ❌ | 🟨 | 🟨 |  | Detector unreliable due to range |
| `A07`   | 2023-08-05T21:00 | 2023-08-05T23:55 | ❌ | ❌ | 🟨 |  | Detector unreliable due to range |
| `A04`   | 2023-08-07T10:55 | 2023-08-07T13:45 | ❌ | ❌ | ❌ |  |  |

**Key**:
- ✅: Complete
- 🟨: In-progress
- ❌: Unavailable

In [ ]:
event_name = "A12-A"
event_data = pd.read_csv(EVENTSPATH).set_index("Event").loc[event_name]
savedir = SAVEPATH / f"{event_name}_{event_data.Start[0:10]}"
savedir.mkdir(exist_ok=True, parents=True)
print(event_data)

### Select Sensor & Channels

Set the `sensor_name` to one of the following and specify the channels you want to work with. Possible selections are:

| Sensor | Channels |
| --- | --- |
| Geosled (`SHRU921_Tetra`) | 0, 1, 2, 3 |
| VLA 1 (`SHRU905_VLA1`) | 0, 1, 2, 3 |
| VLA 2 (`SHRU920_VLA2`) | 0, 1, 2, 3 |

In [6]:
sensor_name = "SHRU921_Tetra"
# sensor_name = "SHRU905_VLA1"
# sensor_name = "SHRU920_VLA2"
channels = [0, 1, 2, 3]

## Load Data

In [ ]:
ds_raw = read_inventory(
    file_path=Path(inv_config[sensor_name]["data"]["destination"]),
    time_start=np.datetime64(event_data["Start"], TIME_PRECISION),
    time_end=np.datetime64(event_data["Stop"], TIME_PRECISION),
    channels=channels,
)
dec_factor = int(ds_raw.stats.sampling_rate // 500)
ds_raw.decimate(dec_factor)
print(f"Decimation factor: {dec_factor}")
print(f"Sampling rate: {ds_raw.stats.sampling_rate}")
print(f"Number of samples: {ds_raw.num_samples}")

### Plot Full Time Series

In [ ]:
fig = plotting.plot_time_series(
    ds_raw,
    title=(
        f"Site {event_data.Site} | {event_data.Start[0:10]} | "
        f"{sensor_name.replace('_', '/')}\nPressure [$\\mu$Pa]"
    ),
)
fig.savefig(
    savedir / f"time-series_{sensor_name}.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="w",
)
plt.show()

### Plot Spectrogram

In [ ]:
fig = plotting.plot_spectrograms(
    ds_raw,
    fmin=10,
    fmax=1000,
    vmin=60,
    vmax=120,
    title=f"Site {event_data.Site} | {event_data.Start[0:10]} | {sensor_name.replace('_', '/')}"
)
fig.savefig(
    savedir / f"spectrogram_{sensor_name}.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="w",
)
plt.show()

## Detection Diagnostics

### Define data pre-processing

In [ ]:
# Filter type
filter_type = "highpass"
# Filter cutoff freq(s) [Hz]
filter_freq = 50.0
# Taper [s]
taper = 5.0
# Pulse Start [ms]
pulse_start = 600
# Pulse End [ms]
pulse_end = 600

ds = ds_raw.copy()

ds.filter(filt_type=filter_type, freq=filter_freq)

### Process channel 0

In [11]:
pulse_start0 = pulse_start
pulse_end0 = pulse_end
stwin0 = 0.025
ltwin0 = 5.0
threshold0 = 3.0
distance0 = 0.95

In [ ]:
fig = plotting.plot_detection_diagnostic(
    ds,
    stwin=stwin0,
    ltwin=ltwin0,
    threshold=threshold0,
    distance=distance0,
    taper=taper,
    sensor=sensor_name,
    channel=0,
)
fig[0].savefig(
    savedir / f"detection-diagnostic_{sensor_name}_chan0.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="w",
)

### Process channel 1

In [13]:
pulse_start1 = pulse_start
pulse_end1 = pulse_end
stwin1 = 0.025
ltwin1 = 5.0
threshold1 = 3.0
distance1 = 0.95

In [ ]:
fig = plotting.plot_detection_diagnostic(
    ds,
    stwin=stwin1,
    ltwin=ltwin1,
    threshold=threshold1,
    distance=distance1,
    taper=taper,
    sensor=sensor_name,
    channel=1,
)
fig[0].savefig(
    savedir / f"detection-diagnostic_{sensor_name}_chan1.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="w",
)
plt.show()

### Process channel 2

In [15]:
pulse_start2 = pulse_start
pulse_end2 = pulse_end
stwin2 = 0.025
ltwin2 = 5.0
threshold2 = 3.0
distance2 = 0.95

In [ ]:
fig = plotting.plot_detection_diagnostic(
    ds,
    stwin=stwin2,
    ltwin=ltwin2,
    threshold=threshold2,
    distance=distance2,
    taper=taper,
    sensor=sensor_name,
    channel=2,
)
fig[0].savefig(
    savedir / f"detection-diagnostic_{sensor_name}_chan2.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="w",
)
plt.show()

### Process channel 3

In [17]:
pulse_start3 = pulse_start
pulse_end3 = pulse_end
stwin3 = 0.025
ltwin3 = 5.0
threshold3 = 3.0
distance3 = 0.95

In [ ]:
fig = plotting.plot_detection_diagnostic(
    ds,
    stwin=stwin3,
    ltwin=ltwin3,
    threshold=threshold3,
    distance=distance3,
    taper=taper,
    sensor=sensor_name,
    channel=3,
)
fig[0].savefig(
    savedir / f"detection-diagnostic_{sensor_name}_chan3.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="w",
)
plt.show()

## Generate TOML text

In [ ]:
event_text = utils.format_event_text(
    event_name, event_data.Site, event_data.Start, event_data.Stop
)
sensor_text = utils.format_sensor_text(
    sensor_name,
    dec_factor,
    filter_type,
    filter_freq,
    taper,
    channels,
    pulse_start=[pulse_start0, pulse_start1, pulse_start2, pulse_start3],
    pulse_end=[pulse_end0, pulse_end1, pulse_end2, pulse_end3],
    stwin=[stwin0, stwin1, stwin2, stwin3],
    ltwin=[ltwin0, ltwin1, ltwin2, ltwin3],
    threshold=[threshold0, threshold1, threshold2, threshold3],
    distance=[distance0, distance1, distance2, distance3],
)
print(event_text)
print(sensor_text)